# Depth Pro 2 on an A100 GPU

<a href="https://colab.research.google.com/github/Cortex78/ml-depth-pro-2/blob/claude/festive-cerf-65vitz/notebooks/depth_pro_2_colab_a100.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook tests [Cortex78/ml-depth-pro-2](https://github.com/Cortex78/ml-depth-pro-2) on an NVIDIA A100 in Google Colab. It:

1. checks the GPU and its driver,
2. installs the repository with uv, as the README does,
3. checks PyTorch, CUDA and the default precision,
4. runs `depth-pro-2 predict` on the example photo with every output: depth, normals, 3D points,
5. runs it on the example video, in both video modes, with normals,
6. reconstructs the example video in 3D: camera poses, a mesh, a point cloud and a COLMAP model,
7. times the four model sizes on a photo and a video clip, with their GPU memory,
8. compares bf16 and fp16 autocast with float32,
9. runs on your own photos and videos, reconstructs your videos, and lets you download it all.

**Before you start:** choose **Runtime → Change runtime type → A100 GPU** (it needs a paid Colab
plan or compute units), then **Runtime → Run all**. The first run downloads the weights: 3.5 GB
for H+, and 1.8 GB more for the other sizes if you benchmark all of them.

The model weights are licensed for research only; see `LICENSE_MODEL` in the repository.

In [ ]:
# @title Settings
REPO_URL = "https://github.com/Cortex78/ml-depth-pro-2.git"  # @param {type:"string"}
BRANCH = "claude/festive-cerf-65vitz"  # @param {type:"string"}
# @markdown Model for the photo, video and precision tests. H+ is the largest and sharpest.
MODEL = "depthpro2-h"  # @param ["depthpro2-s", "depthpro2-b", "depthpro2-l", "depthpro2-h"]
# @markdown Benchmark all four sizes, or only `MODEL`.
BENCHMARK_ALL_MODELS = True  # @param {type:"boolean"}
# @markdown PyTorch build: `auto` keeps PyPI's (CUDA 13) if the driver is 580 or later, else CUDA 12.6.
CUDA_BUILD = "auto"  # @param ["auto", "pypi", "cu126", "cu128", "cu130"]

## 1. GPU and driver

In [ ]:
# @title Check the GPU
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise SystemExit("No NVIDIA GPU in this runtime: Runtime > Change runtime type > A100 GPU.")
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
query = ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
GPU_NAME, GPU_MEMORY, DRIVER = [
    s.strip() for s in subprocess.run(query, capture_output=True, text=True).stdout.splitlines()[0].split(",")
]
print(f"GPU: {GPU_NAME}, {GPU_MEMORY}, driver {DRIVER}")
if "A100" not in GPU_NAME:
    print(f"WARNING: this runtime has a {GPU_NAME}, not an A100. The notebook still runs, but the "
          "timings are not A100 timings. Runtime > Change runtime type > A100 GPU.")

## 2. Install

This clones the branch and runs `uv sync`, which creates `.venv` with the repository's
dependencies. The notebook's own Python is not used for Depth Pro 2: every command below runs in
that environment, through `uv run`.

PyPI's PyTorch for Linux is built for CUDA 13, which needs NVIDIA driver 580 or later. On an older
driver, the next cell points uv at the CUDA 12.6 build of PyTorch instead, the same edit the README
describes for older GPUs on Windows. It only changes the copy cloned here.

In [ ]:
# @title Install uv and the repository
import os
import shutil
from pathlib import Path

%pip install -q uv
REPO_DIR = "/content/ml-depth-pro-2"
if os.path.isdir(f"{REPO_DIR}/.git"):
    # On a rerun, move to the branch's latest commit and undo the previous run's PyTorch edit.
    !git -C {REPO_DIR} fetch -q --depth 1 {REPO_URL} {BRANCH} && git -C {REPO_DIR} reset -q --hard FETCH_HEAD
else:
    !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!git log -1 --format="%h %s (%cr)"

build = CUDA_BUILD
if build == "auto":
    build = "pypi" if int(DRIVER.split(".")[0]) >= 580 else "cu126"
pyproject = Path("pyproject.toml")
text = pyproject.read_text()
if build != "pypi":
    # Take torch and torchvision from the PyTorch index on Linux too, as on Windows.
    linux = "sys_platform == 'win32' or sys_platform == 'linux'"
    for old, new in [
        ("marker = \"sys_platform == 'win32'\"", f'marker = "{linux}"'),
        ("\"torchvision; sys_platform == 'win32'\"", f'"torchvision; {linux}"'),
    ]:
        if old not in text:
            raise RuntimeError(f"pyproject.toml has changed; cannot switch to {build}: {old!r} not found.")
        text = text.replace(old, new)
    text = text.replace("cu130", build)
    pyproject.write_text(text)
print(f"PyTorch build: {build} (driver {DRIVER}).")

os.environ["UV_LINK_MODE"] = "copy"  # /content and uv's cache are on different file systems.
# Colab sets MPLBACKEND to its inline backend, which only its own Python has: the commands below,
# which run in .venv, would fail to import matplotlib. The plots in this notebook use %matplotlib
# inline instead.
os.environ["MPLBACKEND"] = "Agg"
!uv sync

## 3. PyTorch, CUDA and the default precision

In [ ]:
%%writefile /content/check_env.py
import torch

from depth_pro_2.model import default_precision
from depth_pro_2.utils import default_device

print(f"torch {torch.__version__}, built for CUDA {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = default_device()
print(f"default device: {device}")
if device.type != "cuda":
    raise SystemExit("PyTorch cannot use the GPU. Set CUDA_BUILD to cu126 in Settings and rerun from the top.")
major, minor = torch.cuda.get_device_capability(device)
total = torch.cuda.get_device_properties(device).total_memory / 2**30
print(f"GPU: {torch.cuda.get_device_name(device)}, compute capability {major}.{minor}, {total:.0f} GiB")
print(f"bf16 supported natively: {torch.cuda.is_bf16_supported(including_emulation=False)}")
print(f"default precision: {str(default_precision(device)).removeprefix('torch.')}")

In [ ]:
# Stop here, rather than run everything on the CPU, if PyTorch cannot use the GPU.
import subprocess

check = subprocess.run(["uv", "run", "python", "/content/check_env.py"], capture_output=True, text=True)
print(check.stdout, check.stderr[-3000:] if check.returncode else "", sep="")
if check.returncode:
    raise RuntimeError("The environment check failed; see above.")

On an A100 (compute capability 8.0) the default precision is `bfloat16`, as in the paper.

## 4. The example photo

In [ ]:
# @title Predict the example photo with the command line
!uv run depth-pro-2 predict -i assets/example.jpg -o out/ -m {MODEL} --save-all

In [ ]:
# @title Show the photo, its depth and its normals
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

data = np.load("out/example-depth.npz")
print("example-depth.npz holds:", {key: data[key].shape for key in data})
print("intrinsics:")
print(data["intrinsics"])
depth, focal = data["depth"], float(data["focallength_px"])
finite = depth[np.isfinite(depth)]
print(f"depth {depth.shape}, focal length {focal:.1f} px")
print(f"finite depth: {finite.min():.2f} m to {finite.max():.2f} m, median {np.median(finite):.2f} m; "
      f"{100 * (1 - finite.size / depth.size):.1f}% infinitely far")

fig, axes = plt.subplots(2, 2, figsize=(18, 13))
axes = axes.ravel()
axes[0].imshow(Image.open("assets/example.jpg"))
axes[0].set_title("input")
axes[1].imshow(Image.open("out/example-depth.png"))
axes[1].set_title("example-depth.png (red near, blue far)")
axes[2].imshow(Image.open("out/example-normals.png"))
axes[2].set_title("example-normals.png (red right, green up, blue toward the camera)")
shown = axes[3].imshow(np.clip(depth, 0, np.percentile(finite, 95)), cmap="turbo_r")
axes[3].set_title("depth in meters, clipped at the 95th percentile")
fig.colorbar(shown, ax=axes[3], label="m", fraction=0.035)
for ax in axes:
    ax.axis("off")
plt.show()
!ls -lh out/

## 5. The example video, both modes

In [ ]:
# @title Buffered mode (the default): overlapping clips of 16 frames
!uv run depth-pro-2 predict -i assets/example.mp4 -o out/ -m {MODEL} --save-normals
!mv out/example-depth.mp4 out/example-depth-buffered.mp4

In [ ]:
# @title Unbuffered mode: each frame from itself and the 15 before it, as for a live camera
!uv run depth-pro-2 predict -i assets/example.mp4 -o out/ -m {MODEL} --mode unbuffered
!mv out/example-depth.mp4 out/example-depth-unbuffered.mp4

In [ ]:
# @title Play both previews
from IPython.display import HTML, Video, display

for mode in ("buffered", "unbuffered"):
    display(HTML(f"<b>{mode}</b>"))
    display(Video(f"out/example-depth-{mode}.mp4", embed=True, width=640))
display(HTML("<b>normals (buffered)</b>"))
display(Video("out/example-normals.mp4", embed=True, width=640))

## 6. 3D reconstruction of the example video

`depth-pro-2 reconstruct` turns a video of a static scene into a 3D model: each frame's metric
depth makes it an RGB-D image; SIFT matches lifted to 3D by that depth give the camera poses
(PnP with RANSAC, then a pose graph); and a TSDF volume fuses the frames into a mesh. With
`--colmap`, it also writes a COLMAP model of the keyframes, to refine in COLMAP or to train
Gaussian splatting from. See the README's "3D reconstruction" section.

In [ ]:
# @title Install the reconstruction extra (Open3D and OpenCV)
!apt-get -qq install -y libegl1 libgl1 libgomp1 > /dev/null
!uv sync --extra reconstruct

In [ ]:
# @title Reconstruct the example video
!uv run --extra reconstruct depth-pro-2 reconstruct -i assets/example.mp4 -o recon/ -m {MODEL} --colmap
!ls -lh recon/

In [ ]:
%%writefile /content/cloud_sample.py
# A random sample of a PLY point cloud, as npz, for plotting.
import sys

import numpy as np
import open3d as o3d

cloud = o3d.io.read_point_cloud(sys.argv[1])
points, colors = np.asarray(cloud.points), np.asarray(cloud.colors)
count = int(sys.argv[3])
if len(points) > count:
    keep = np.random.default_rng(0).choice(len(points), count, replace=False)
    points, colors = points[keep], colors[keep]
np.savez(sys.argv[2], points=points.astype(np.float32), colors=colors.astype(np.float32))

In [ ]:
# @title Show the point cloud and the camera path in 3D (drag to rotate)
import json
import shlex

import numpy as np
import plotly.graph_objects as go


def show_reconstruction(prefix, count=120_000):
    """Plot a reconstruction's fused cloud, a sample of it, and its camera path."""
    cloud = shlex.quote(f"{prefix}-cloud.ply")
    !uv run --extra reconstruct python /content/cloud_sample.py {cloud} /content/cloud.npz {count}
    sample = np.load("/content/cloud.npz")
    points, colors = sample["points"], (sample["colors"] * 255).astype(int)
    cameras = json.load(open(f"{prefix}-cameras.json"))
    path = np.array([np.array(f["camera_to_world"])[:3, 3] for f in cameras["frames"]])
    # Camera frame: x right, y down, z forward. Plot x, z and -y, so up is up.
    figure = go.Figure([
        go.Scatter3d(
            x=points[:, 0], y=points[:, 2], z=-points[:, 1], mode="markers",
            marker=dict(size=1.2, color=[f"rgb({r},{g},{b})" for r, g, b in colors]), name="cloud",
        ),
        go.Scatter3d(
            x=path[:, 0], y=path[:, 2], z=-path[:, 1], mode="lines+markers",
            marker=dict(size=3, color="red"), line=dict(color="red", width=4), name="camera",
        ),
    ])
    figure.update_layout(
        height=700, scene=dict(aspectmode="data", xaxis_title="x (m)", yaxis_title="forward (m)",
                               zaxis_title="up (m)"),
        margin=dict(l=0, r=0, t=30, b=0),
    )
    figure.show()
    extent = path.max(0) - path.min(0)
    print(f"{len(path)} keyframes; camera moved within {extent.round(2)} m; {len(points)} points shown")


show_reconstruction("recon/example")

## 7. Speed and memory of each model size

Times one photo (the median of 5 runs, after a warm-up) and the 64 frames of the example video in
both modes, at full resolution (1920×1440), in each model's default precision. All weights are
downloaded first, so `load_s` is the time to build the model and load its weights. Frames are
decoded and moved to the GPU before timing. Peak memory is what PyTorch allocated, the clip's
frames included. A run that does not fit in the GPU's memory is reported as out of memory instead
of stopping the notebook.

In [ ]:
%%writefile /content/bench.py
"""Time Depth Pro 2 on a photo and a video clip, per model and precision, on one GPU."""

import argparse
import gc
import json
import time

import av
import torch

from depth_pro_2 import create_model_and_transforms, load_rgb, upright
from depth_pro_2.model import PRESETS, default_precision, resolve_checkpoint

PRECISIONS = {"bf16": torch.bfloat16, "fp16": torch.float16, "fp32": torch.float32}

parser = argparse.ArgumentParser()
parser.add_argument(
    "--models", nargs="+", default=["depthpro2-s", "depthpro2-b", "depthpro2-l", "depthpro2-h"]
)
parser.add_argument("--precisions", nargs="+", default=["default"], choices=["default", *PRECISIONS])
parser.add_argument("--image", default="assets/example.jpg")
parser.add_argument("--video", default="assets/example.mp4")
parser.add_argument("--frames", type=int, default=64, help="Video frames to time, at most.")
parser.add_argument("--window", type=int, default=16)
parser.add_argument("--overlap", type=int, default=4)
parser.add_argument("--repeats", type=int, default=5, help="Timed photo runs, after one warm-up.")
parser.add_argument("--device", default="cuda")
parser.add_argument("--out", default="bench.jsonl")
args = parser.parse_args()

device = torch.device(args.device)
cuda = device.type == "cuda"


def sync():
    if cuda:
        torch.cuda.synchronize(device)


def reset_peak():
    if cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)


def peak_gb():
    return torch.cuda.max_memory_allocated(device) / 2**30 if cuda else float("nan")


image, f_px = load_rgb(args.image)
with av.open(args.video) as container:
    stream = container.streams.video[0]
    raw_frames = []
    for frame in container.decode(stream):
        raw_frames.append(upright(frame))
        if len(raw_frames) == args.frames:
            break

# Download every checkpoint and start CUDA first, so neither counts in a model's load time.
for name in args.models:
    resolve_checkpoint(PRESETS[name].checkpoint_uri)
    resolve_checkpoint(PRESETS[name].fov_checkpoint_uri)
torch.zeros(1, device=device)
sync()

out = open(args.out, "w")  # One row per line as soon as it is done, so a crash keeps the others.
for name in args.models:
    for precision_name in args.precisions:
        precision = None if precision_name == "default" else PRECISIONS[precision_name]
        reset_peak()
        start = time.perf_counter()
        model, transform = create_model_and_transforms(name, device, precision)
        sync()
        load_s = time.perf_counter() - start
        row = {
            "model": name,
            "precision": str(model.precision).removeprefix("torch."),
            "load_s": round(load_s, 1),
        }

        # A photo, on its own, with the focal length from EXIF if it has one.
        row["photo"] = f"{image.shape[1]}x{image.shape[0]}"
        try:
            x = transform(image)
            model.infer(x, f_px=f_px)  # Warm-up: cuDNN autotuning and lazy CUDA init.
            sync()
            reset_peak()
            times = []
            for _ in range(args.repeats):
                start = time.perf_counter()
                prediction = model.infer(x, f_px=f_px)
                sync()
                times.append(time.perf_counter() - start)
            times.sort()
            depth = prediction["depth"].float()
            finite = depth[torch.isfinite(depth)]
            row |= {
                "photo_ms": round(1000 * times[len(times) // 2], 1),
                "photo_peak_gb": round(peak_gb(), 2),
                "photo_median_depth_m": round(float(finite.median()), 3) if finite.numel() else None,
                "focal_px": round(float(prediction["focallength_px"]), 1),
            }
            del x, prediction, depth, finite
        except torch.OutOfMemoryError:
            row["photo_ms"] = "out of memory"
        gc.collect()
        reset_peak()

        # A video clip, both modes. Frames go to the GPU first, so decoding is not timed.
        row["video"] = f"{len(raw_frames)} frames {raw_frames[0].shape[1]}x{raw_frames[0].shape[0]}"
        frames = [transform(f) for f in raw_frames]
        for mode in ("buffered", "unbuffered"):
            try:
                warmup = frames[: args.window]
                list(model.infer_video(warmup, args.window, args.overlap, mode=mode))
                sync()
                reset_peak()
                start = time.perf_counter()
                count = sum(1 for _ in model.infer_video(frames, args.window, args.overlap, mode=mode))
                sync()
                elapsed = time.perf_counter() - start
                row[f"{mode}_fps"] = round(count / elapsed, 2)
                row[f"{mode}_peak_gb"] = round(peak_gb(), 2)
            except torch.OutOfMemoryError:
                row[f"{mode}_fps"] = "out of memory"
            gc.collect()
            reset_peak()

        print(json.dumps(row), flush=True)
        out.write(json.dumps(row) + "\n")
        out.flush()
        del model, transform, frames
        gc.collect()
        reset_peak()

out.close()
gpu = torch.cuda.get_device_name(device) if cuda else "CPU"
print(f"Default precision on {gpu}: {str(default_precision(device)).removeprefix('torch.')}.")

In [ ]:
# @title Run the benchmark
models = "depthpro2-s depthpro2-b depthpro2-l depthpro2-h" if BENCHMARK_ALL_MODELS else MODEL
!rm -f /content/bench.jsonl
!uv run python /content/bench.py --models {models} --out /content/bench.jsonl

In [ ]:
# @title Results
import json

import pandas as pd

path = "/content/bench.jsonl"
rows = [json.loads(line) for line in open(path)] if os.path.exists(path) else []
if rows:
    print(f"{GPU_NAME}, {GPU_MEMORY}; photo {rows[0]['photo']}, video {rows[0]['video']}")
    display(pd.DataFrame(rows).set_index("model"))
else:
    print("No benchmark results: the benchmark cell above failed; see its output.")

## 8. Precision: bf16 and fp16 against float32

Runs the example photo three times with `MODEL`, under bf16 autocast, fp16 autocast and in float32,
and reports how far the depth of each autocast run is from float32's.

In [ ]:
%%writefile /content/precision_check.py
"""Compare a photo's depth under bf16 and fp16 autocast with float32, for one model."""

import argparse

import torch

from depth_pro_2 import create_model_and_transforms, load_rgb

parser = argparse.ArgumentParser()
parser.add_argument("--model", default="depthpro2-h")
parser.add_argument("--image", default="assets/example.jpg")
parser.add_argument("--device", default="cuda")
args = parser.parse_args()

image, f_px = load_rgb(args.image)
depths = {}
for dtype in (torch.float32, torch.bfloat16, torch.float16):
    model, transform = create_model_and_transforms(args.model, args.device, dtype)
    prediction = model.infer(transform(image), f_px=f_px)
    depths[dtype] = (prediction["depth"].float(), float(prediction["focallength_px"]))
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

reference, reference_f = depths[torch.float32]
print(f"{args.model} on {args.image}, against float32 (focal {reference_f:.1f} px):")
for dtype in (torch.bfloat16, torch.float16):
    depth, f = depths[dtype]
    both = torch.isfinite(depth) & torch.isfinite(reference)
    rel = ((depth[both] - reference[both]).abs() / reference[both]).float()
    sky = (torch.isfinite(depth) != torch.isfinite(reference)).float().mean()
    print(
        f"  {str(dtype).removeprefix('torch.'):>9}: median relative depth difference "
        f"{100 * rel.median():.3f}%, 99th percentile {100 * rel.quantile(0.99):.2f}%, "
        f"focal {f:.1f} px, pixels finite in one only {100 * sky:.3f}%"
    )

In [ ]:
!uv run python /content/precision_check.py --model {MODEL}

## 9. Your own photos and videos

Upload photos (`.jpg`, `.png`, `.heic`, ...) or videos (`.mp4`, `.mov`, ...). They are predicted
with `MODEL`, each photo on its own and each video as a whole. Photos get every output (depth in
meters as `.npz`, normals, a point cloud); videos get the color preview and the normals video, since
an `.npz` or `.ply` per frame quickly takes tens of GB. Each run of the cell predicts only the files
uploaded in that run. The next cell then reconstructs the uploaded videos in 3D.

In [ ]:
# @title Upload and predict
from google.colab import files

import shlex

VIDEOS = (".mov", ".mp4", ".m4v", ".avi", ".mkv", ".webm", ".mpg", ".mpeg", ".mts")
os.makedirs("uploads", exist_ok=True)
photos, videos = [], []
for name in files.upload():
    path = os.path.join("uploads", name)
    shutil.move(name, path)
    (videos if name.lower().endswith(VIDEOS) else photos).append(path)
if photos:
    inputs = " ".join(f"-i {shlex.quote(p)}" for p in photos)
    !uv run depth-pro-2 predict {inputs} -o out_uploads/ -m {MODEL} --save-all
if videos:
    inputs = " ".join(f"-i {shlex.quote(p)}" for p in videos)
    !uv run depth-pro-2 predict {inputs} -o out_uploads/ -m {MODEL} --save-normals

In [ ]:
# @title Show the results
from IPython.display import Image as ShowImage

for name in sorted(os.listdir("out_uploads")) if os.path.isdir("out_uploads") else []:
    path = os.path.join("out_uploads", name)
    if name.endswith(("-depth.png", "-normals.png")):
        display(HTML(f"<b>{name}</b>"), ShowImage(path, width=640))
    elif name.endswith(("-depth.mp4", "-normals.mp4")):
        display(HTML(f"<b>{name}</b>"), Video(path, embed=True, width=640))

In [ ]:
# @title Reconstruct the uploaded videos in 3D
# @markdown Raise `STRIDE` for videos longer than about 30 seconds: every STRIDE-th frame is a keyframe.
STRIDE = 1  # @param {type:"integer"}
for video in videos:
    !uv run --extra reconstruct depth-pro-2 reconstruct -i {shlex.quote(video)} -o recon_uploads/ -m {MODEL} --stride {STRIDE} --colmap
    show_reconstruction(f"recon_uploads/{Path(video).stem}")

## 10. Download the results

In [ ]:
# @title Zip and download every output
from google.colab import files

if os.path.exists("/content/bench.jsonl"):
    shutil.copy("/content/bench.jsonl", "out/")
folders = " ".join(f for f in ("out", "out_uploads", "recon", "recon_uploads") if os.path.isdir(f))
!zip -qr /content/depth_pro_2_results.zip {folders}
!ls -lh /content/depth_pro_2_results.zip
files.download("/content/depth_pro_2_results.zip")